# Convert the field `tags` to an array and look for spark related tags 

In this notebook you will solve one problem using higher order functions.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, split, expr, transform, length, upper, size, filter, lit
)

import os

In [2]:
spark = (
    SparkSession
    .builder
    .appName('Higher Order Functions I')
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 14:38:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

questions_json_input_path = os.path.join(project_path, 'data/questions-json')

#### Read the questions data from JSON:

In [4]:
questionsDF = (
    spark
    .read
    .format('json')
    .option('path', questions_json_input_path)
    .load()
)

#### Transform tags - first use SQL expression:

Hint:
* check the tags column, it is in this format `<tag1><tag2><tag3>`
* convert the tags into an array: [tag1, tag2, tag3, ...]
  * use [substr](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.substr.html) and length to get rid of the first and last angle bracket 
* then split the string to an array
 * use [split](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html#pyspark.sql.functions.split)
* after having the array, use higher order function filter to keep in the arrays only spark-related tags
  * use [filter](https://spark.apache.org/docs/latest/api/sql/index.html#filter)
  * to identify spark related tags use [like](https://spark.apache.org/docs/latest/api/sql/index.html#like) '%spark%'
  * filter out empty tags

In [5]:
(
    questionsDF
    .withColumn('tags', col('tags').substr(lit(2), length('tags') - 2))
    .withColumn('tags', split('tags', '><'))
    .withColumn('tags', expr("FILTER(tags, value -> value like '%spark%')"))
    .filter(size(col('tags')) > 0)
    .select('question_id', 'title', 'tags')
).show(truncate=70, n=10)

+-----------+----------------------------------------------------------------------+-------------------------------------------+
|question_id|                                                                 title|                                       tags|
+-----------+----------------------------------------------------------------------+-------------------------------------------+
|   59845157|              How to include non-aggregated columns after aggregation?|           [apache-spark, apache-spark-sql]|
|   61155679|How to write custom function from percentile_approx code which give...|           [apache-spark, apache-spark-sql]|
|   61838509|           How to do aggregate functions, may columns and extract back|           [apache-spark, apache-spark-sql]|
|   61226052|             how to parallellize this in spark using spark dataset api|           [apache-spark, apache-spark-sql]|
|   59669880|how to populate select clause of dataframe dynamically? giving Anal...|           [a

#### Do the same with PySpark DSL

* Available since Spark 3.1
* use
  * [filter](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.filter.html)
  * [like](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.like.html) 

In [6]:
(
    questionsDF
    .withColumn('tags', col('tags').substr(lit(2), length('tags') - 2))
    .withColumn('tags', split('tags', '><'))
    .withColumn('tags', filter(col('tags'), lambda x: x.like('%spark%')))
    .filter(size(col('tags')) > 0)
    .select('question_id', 'title', 'tags')
).show(truncate=70, n=10)

+-----------+----------------------------------------------------------------------+-------------------------------------------+
|question_id|                                                                 title|                                       tags|
+-----------+----------------------------------------------------------------------+-------------------------------------------+
|   59845157|              How to include non-aggregated columns after aggregation?|           [apache-spark, apache-spark-sql]|
|   61155679|How to write custom function from percentile_approx code which give...|           [apache-spark, apache-spark-sql]|
|   61838509|           How to do aggregate functions, may columns and extract back|           [apache-spark, apache-spark-sql]|
|   61226052|             how to parallellize this in spark using spark dataset api|           [apache-spark, apache-spark-sql]|
|   59669880|how to populate select clause of dataframe dynamically? giving Anal...|           [a

To read more information about HOFs see my [article](https://towardsdatascience.com/higher-order-functions-with-spark-3-1-7c6cf591beaa)

In [ ]:
spark.stop()